In [1]:

# Standard Libraries

import os
import gc
import random
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


# Data Manipulation

import numpy as np
import pandas as pd


# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("Set2")


# Machine Learning

from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Miscellaneous

from tqdm import tqdm

# Production-Ready Data Preprocessing Pipeline

## Pipeline Overview

```mermaid
flowchart LR
    A[Raw Data] --> B[Time-aware Split]
    B --> C[Separate Features & Target]
    C --> D[Train-only Fit]

    D --> E[Numeric Imputer]
    D --> F[Missing Indicators]
    D --> G[Categorical Imputer]
    D --> H[Frequency Encoder]
    D --> I[One-Hot Encoder]
    D --> J[Standard Scaler]

    E --> K[Processed Training Set]
    F --> K
    G --> K
    H --> K
    I --> K
    J --> K

    K --> L[Validation/Test<br/>Transform Only]
```

The most important principle is:

> **Nothing is learned from the validation or test data.**


All preprocessing parameters (medians, category mappings, scaling statistics) are fit **only on the training set** to avoid data leakage. A chronological split based on `TransactionDT` better reflects how fraud detection systems operate in production.



# 4.1 Sort by Transaction Time

Before splitting, ensure observations are in chronological order.

In [4]:
DATA_DIR =  Path("data")

data = pd.read_csv(DATA_DIR/"raw/cleaned_df.csv")

In [6]:
# Sort Chronologically


data = data.sort_values("TransactionDT").reset_index(drop=True)

print(data.shape)

(590540, 425)


# 4.2 Time-based Train / Validation / Test Split

Instead of a random split, we'll simulate deployment.

**70% Training**

**15% Validation**

**15% Testing**

In [7]:
# Chronological Split

n = len(data)

train_end = int(n * 0.70)

valid_end = int(n * 0.85)

train_df = data.iloc[:train_end]

valid_df = data.iloc[train_end:valid_end]

test_df = data.iloc[valid_end:]

In [8]:
print("Training :", train_df.shape)

print("Validation :", valid_df.shape)

print("Testing :", test_df.shape)

Training : (413378, 425)
Validation : (88581, 425)
Testing : (88581, 425)


A chronological split prevents future transactions from influencing training and better approximates real deployment, where models always predict on unseen future activity.


# 4.3 Separate Features and Labels

In [9]:
TARGET = "isFraud"

DROP_COLUMNS = [
    "TransactionID",
    TARGET
]

In [10]:
X_train = train_df.drop(columns=DROP_COLUMNS)

y_train = train_df[TARGET]

X_valid = valid_df.drop(columns=DROP_COLUMNS)

y_valid = valid_df[TARGET]

X_test = test_df.drop(columns=DROP_COLUMNS)

y_test = test_df[TARGET]

# 4.4 Detect Feature Types

Always detect feature types **after** splitting.

In [11]:
numeric_features = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(include="object")
    .columns
    .tolist()
)

print("Numeric Features :", len(numeric_features))

print("Categorical Features :", len(categorical_features))

Numeric Features : 394
Categorical Features : 29


# 4.5 Missing Value Indicators

Missing values can be predictive.

Instead of only imputing them, we'll create indicator variables.

In [12]:
# Missing Indicator Features

indicator_columns = []

for col in X_train.columns:

    if X_train[col].isnull().mean() > 0.10:  # indicate if more that 10%

        indicator_columns.append(col)

print(f"Indicator Features : {len(indicator_columns)}")

Indicator Features : 313


In [13]:
indicator_columns[:5] # print head incator columns

['addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain']

In [14]:
def add_missing_indicators(df, cols):

    df = df.copy()

    for col in cols:

        df[col + "_missing"] = (
            df[col]
            .isnull()
            .astype(np.int8)
        )

    return df

In [15]:
X_train = add_missing_indicators(X_train,
                                 indicator_columns)

X_valid = add_missing_indicators(X_valid,
                                 indicator_columns)

X_test = add_missing_indicators(X_test,
                                indicator_columns)

Using explicit missing indicators is a common approach in fraud detection because the absence of information (for example, missing identity fields) can itself carry predictive value.




# 4.6 Numerical Imputer

We'll use the **median** because transaction-related variables are often highly skewed.

In [16]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(
    strategy="median"
)

numeric_imputer.fit(
    X_train[numeric_features]
)

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](394,)","['TransactionDT','TransactionAmt','card1',...,'id_19','id_20','id_32']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,394
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[float64](394,)","[4748457.5 , 68.95, 9633. ,..., 341. , 469. , 24. ]"


In [17]:
X_train[numeric_features] = numeric_imputer.transform(
    X_train[numeric_features]
)

X_valid[numeric_features] = numeric_imputer.transform(
    X_valid[numeric_features]
)

X_test[numeric_features] = numeric_imputer.transform(
    X_test[numeric_features]
)

# 4.7 Categorical Imputer

Never fill categories with the mode.

Create a new category called **"Missing"**.

In [18]:
categorical_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Missing"
)

categorical_imputer.fit(
    X_train[categorical_features]
)

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",'Missing'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](29,)","['ProductCD','card4','card6',...,'id_38','DeviceType','DeviceInfo']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,29
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[object](29,)","['Missing','Missing','Missing',...,'Missing','Missing','Missing']"


In [19]:
X_train[categorical_features] = (
    categorical_imputer.transform(
        X_train[categorical_features]
    )
)

X_valid[categorical_features] = (
    categorical_imputer.transform(
        X_valid[categorical_features]
    )
)

X_test[categorical_features] = (
    categorical_imputer.transform(
        X_test[categorical_features]
    )
)

This preserves the distinction between an observed category and a genuinely missing value, which can be informative in fraud datasets.




# 4.8 Frequency Encoding

One-Hot Encoding hundreds of categories is inefficient.

Instead, we'll frequency encode high-cardinality features.

In [20]:
def frequency_encode(train, valid, test, columns):

    train = train.copy()

    valid = valid.copy()

    test = test.copy()

    for col in columns:

        freq = (
            train[col]
            .value_counts(dropna=False)
        )

        train[col] = train[col].map(freq)

        valid[col] = valid[col].map(freq)

        test[col] = test[col].map(freq)

    return train, valid, test

In [21]:
high_cardinality = [
    col
    for col in categorical_features
    if X_train[col].nunique() > 10
]

low_cardinality = [
    col
    for col in categorical_features
    if X_train[col].nunique() <= 10
]

print("High Cardinality :", len(high_cardinality))

print("Low Cardinality :", len(low_cardinality))

High Cardinality : 6
Low Cardinality : 23


In [22]:
X_train, X_valid, X_test = frequency_encode(
    X_train,
    X_valid,
    X_test,
    high_cardinality
)

Frequency encoding is widely used for IEEE-CIS because it preserves information from high-cardinality variables such as card identifiers and email domains without exploding the feature space.





# 4.9 One-Hot Encoding

Low-cardinality variables are still suitable for one-hot encoding.

In [23]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(
    X_train[low_cardinality]
)

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [24]:
train_ohe = encoder.transform(
    X_train[low_cardinality]
)

valid_ohe = encoder.transform(
    X_valid[low_cardinality]
)

test_ohe = encoder.transform(
    X_test[low_cardinality]
)

In [29]:
X_train.head()

,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,id_31_missing,id_32_missing,id_33_missing,id_34_missing,id_35_missing,id_36_missing,id_37_missing,id_38_missing,DeviceType_missing,DeviceInfo_missing
0,86400.0,68.5,W,13926.0,361.0,150.0,discover,142.0,credit,315.0,...,1,1,1,1,1,1,1,1,1,1
1,86401.0,29.0,W,2755.0,404.0,150.0,mastercard,102.0,credit,325.0,...,1,1,1,1,1,1,1,1,1,1
2,86469.0,59.0,W,4663.0,490.0,150.0,visa,166.0,debit,330.0,...,1,1,1,1,1,1,1,1,1,1
3,86499.0,50.0,W,18132.0,567.0,150.0,mastercard,117.0,debit,476.0,...,1,1,1,1,1,1,1,1,1,1
4,86506.0,50.0,H,4497.0,514.0,150.0,mastercard,102.0,credit,420.0,...,0,0,0,0,0,0,0,0,0,0


In [41]:
X_train.to_csv(DATA_DIR/'preprocessed/X_train.csv', index=False)
X_valid.to_csv(DATA_DIR/'preprocessed/X_valid.csv', index=False)
X_test.to_csv(DATA_DIR/'preprocessed/X_test.csv', index=False)

y_train.to_csv(DATA_DIR/'preprocessed/y_train.csv', index=False)
y_valid.to_csv(DATA_DIR/'preprocessed/y_valid.csv', index=False)
y_test.to_csv(DATA_DIR/'preprocessed/y_test.csv', index=False)
# print("Saved Preprocessed data.")